In [105]:
import chipwhisperer as cw
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error 
from scipy import stats
import sys
sys.path.append("/home/40265864@ecit.qub.ac.uk/Toeplitz_attack/")
from privacy_amplification.FFT.FFT import *

SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_SAM4S'
SS_VER = 'SS_VER_2_1'

try:
    if not scope.connectStatus:
        scope.con()
except NameError:
    scope = cw.scope()

try:
    if SS_VER == "SS_VER_2_1":
        target_type = cw.targets.SimpleSerial2
    elif SS_VER == "SS_VER_2_0":
        raise OSError("SS_VER_2_0 is deprecated. Use SS_VER_2_1")
    else:
        target_type = cw.targets.SimpleSerial
except:
    SS_VER="SS_VER_1_1"
    target_type = cw.targets.SimpleSerial

try:
    target = cw.target(scope, target_type)
except:
    print("INFO: Caught exception on reconnecting to target - attempting to reconnect to scope first.")
    print("INFO: This is a work-around when USB has died without Python knowing. Ignore errors above this line.")
    scope = cw.scope()
    target = cw.target(scope, target_type)


print("INFO: Found ChipWhisperer😍")

if "STM" in PLATFORM or PLATFORM == "CWLITEARM" or PLATFORM == "CWNANO":
    prog = cw.programmers.STM32FProgrammer
elif PLATFORM == "CW303" or PLATFORM == "CWLITEXMEGA":
    prog = cw.programmers.XMEGAProgrammer
elif "neorv32" in PLATFORM.lower():
    prog = cw.programmers.NEORV32Programmer
elif PLATFORM == "CW308_SAM4S" or PLATFORM == "CWHUSKY":
    prog = cw.programmers.SAM4SProgrammer
else:
    prog = None

INFO: Caught exception on reconnecting to target - attempting to reconnect to scope first.
INFO: This is a work-around when USB has died without Python knowing. Ignore errors above this line.
INFO: Found ChipWhisperer😍


In [109]:
import time
time.sleep(0.05)
scope.default_setup()

def reset_target(scope):
    if PLATFORM == "CW303" or PLATFORM == "CWLITEXMEGA":
        scope.io.pdic = 'low'
        time.sleep(0.1)
        scope.io.pdic = 'high_z' #XMEGA doesn't like pdic driven high
        time.sleep(0.1) #xmega needs more startup time
    elif "neorv32" in PLATFORM.lower():
        raise IOError("Default iCE40 neorv32 build does not have external reset - reprogram device to reset")
    elif PLATFORM == "CW308_SAM4S" or PLATFORM == "CWHUSKY":
        scope.io.nrst = 'low'
        time.sleep(0.25)
        scope.io.nrst = 'high_z'
        time.sleep(0.25)
    else:  
        scope.io.nrst = 'low'
        time.sleep(0.05)
        scope.io.nrst = 'high_z'
        time.sleep(0.05)

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.clock.clkgen_freq                  changed from 0                         to 7370129.87012987         
scope.clock.adc_freq                     changed from 0                         to 29480519.48051948        
scope.clock.extclk_monitor_enabled       changed from True                      to False                    
scope.clock.extclk_tolerance             changed from 1144409.1796875           to 13096723.705530167       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2       

In [110]:
scope.adc.samples = 5000                 # Number of samples per segment
#scope.adc.stream_mode = "segmented"            # Enable segmented capture
#scope.adc.segments = 12          # Total number of segments to capture
scope.adc.offset = 0

# Gain settings
#scope.gain.db = 30

# Trigger settings
#scope.trigger.triggers = "tio4"          # Make sure this matches your trigger pin
#scope.clock.clkgen_freq = 7370000        # Or whatever frequency your target uses
#scope.clock.adc_src = "clkgen_x4"

In [111]:
%%bash -s "$PLATFORM" "$SS_VER"
cd firmware/Toeplitz_FFT
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
.
arm-none-eabi-gcc (15:10.3-2021.07-4) 10.3.1 20210621 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

Welcome to another exciting ChipWhisperer target build!!
Size after:
+--------------------------------------------------------
+ Built for platform Microchip SAM4S with:
   text	   data	    bss	    dec	    hex	filename
  24180	    108	   4140	  28428	   6f0c	Toeplitz_FFT-CW308_SAM4S.elf
+ CRYPTO_TARGET = NONE
+ CRYPTO_OPTIONS = AES128C
+--------------------------------------------------------


In [112]:
cw.program_target(scope, prog, "firmware/Toeplitz_FFT/Toeplitz_FFT-{}.hex".format(PLATFORM))

In [6]:
target.write("00010101110101011011\n")

In [7]:
target.write("1010110101011\n")

In [8]:
target.write("01101101111000011100\n")


In [74]:
reset_target(scope)

In [113]:
def get_trace(x):
    #num_char = target.in_waiting()
    #while num_char > 0:
        #target.read(num_char, 10)
        #time.sleep(0.01)
        #num_char = target.in_waiting()
    time.sleep(0.1)
    #target.flush()
    scope.arm()
    target.write(x)
    if scope.capture():
        raise RuntimeError("Capture failed")
    trace_segments = scope.get_last_trace()#_segmented()
    return trace_segments


In [115]:
traces = []
reset_target(scope)

##inital toeplitz setup
target.write("00010\n")
time.sleep(0.01)
target.write("1010\n")


trace = get_trace("00001\n")
k=7

(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:732) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 13
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:732) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 12


RuntimeError: Capture failed

In [63]:
fig = cw.plot()
for seg in trace[0:3]:
    fig *= cw.plot(seg)
fig

:Overlay
   .Curve.I   :Curve   [x]   (y)
   .Curve.II  :Curve   [x]   (y)
   .Curve.III :Curve   [x]   (y)
   .Curve.IV  :Curve   [x]   (y)

In [ ]:
traces = []
reset_target(scope)

##inital toeplitz setup
target.write("00010\n")
time.sleep(0.01)
target.write("1010\n")


x1 = "00000\n"
x2 = "11111\n"
x3 = "01010\n"
x4 = "11111\n"
x5 = "00000\n"
#x6 = "00000000000111111111\n"
x6 = "10100\n"



for i in range(4):
    warmup = get_trace(x2)

    

for i in range(10):
    traces.append(get_trace(x1))
for i in range(10):
    traces.append(get_trace(x2))
for i in range(10):
    traces.append(get_trace(x3))
for i in range(10):
    traces.append(get_trace(x4))
for i in range(10):
    traces.append(get_trace(x5))
for i in range(10):
    traces.append(get_trace(x6))

(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:732) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 10
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:732) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 10
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:732) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 10
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:732) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is 

KeyboardInterrupt: 

In [70]:
fig = cw.plot()
#for i in range(len(traces)):
fig *= cw.plot(traces[0][0])
fig

:Overlay
   .Curve.I  :Curve   [x]   (y)
   .Curve.II :Curve   [x]   (y)

In [29]:
avg1 = np.mean(traces[0:10],axis=0)
avg2 = np.mean(traces[10:20],axis=0)
avg3 = np.mean(traces[20:30],axis=0)
avg4 = np.mean(traces[30:40],axis=0)
avg5 = np.mean(traces[40:50],axis=0)
avg6 = np.mean(traces[50:60],axis=0)



fig = cw.plot()
fig *= cw.plot(avg6-avg5)
fig

:Overlay
   .Curve.I  :Curve   [x]   (y)
   .Curve.II :Curve   [x]   (y)

In [17]:
fig = cw.plot()
fig *= cw.plot(traces[45] - traces[55])
fig

NameError: name 'traces' is not defined

In [61]:
def gen_hyp_string_basic(x):
    x_str = list("0000000000\n")
    x_bin = bin(x)[2:]
    x_bin = x_bin[::-1]
    for i in range(len(x_bin)):
        x_str[int(i)] = x_bin[i]
    x_str = ''.join(x_str)
    return x_str

1. Set a random key to be the correct key & record trace as reference trace
2. For the first 16 bits of the re-ordered key input, make a guess
3. Account for padded bits to reduce key space
4. For each guess, carry out FFT and record trace
5. Compare each guess to reference trace, select max correlation as guess

In [60]:
##let x be a decimal input for the bits 0,2,4,6,8,10,12,14,16,18
def gen_hyp_string_8_bit_chunk(x, offset):
    x_str = list("00000000000000000000\n")
    x_bin = bin(x)[2:]
    x_bin = x_bin[::-1]
    for i in range(len(x_bin)):
        x_str[int(4*i + offset)] = x_bin[i] ##change 1 to 4 in real scenario
    x_str = ''.join(x_str)
    return x_str

In [8]:
x = gen_hyp_string(228,0)
print(x)

IndexError: list assignment index out of range

In [68]:
def make_guesses(offset):
    traces = []
    reset_target(scope)

    ##inital toeplitz setup
    #target.write("00010101110101011011\n")
    target.write("0110100110\n")

    time.sleep(0.01)
    #target.write("1010110101011\n")
    target.write("1010111\n")


    x2 = "11111\n"


    for i in range(4):
        warmup = get_trace(x2)


    for i in range(1024):
        if i % 200 == 0:
            target.flush()
            reset_target(scope)

            target.write("0110100110\n")

            time.sleep(0.01)
            #target.write("1010110101011\n")
            target.write("1010111\n")


            x2 = "11111\n"


            for i in range(4):
                warmup = get_trace(x2)
        print("Testing Hyptohesis {0}".format(i))
        x = gen_hyp_string_basic(i)
        traces.append(get_trace(x))

    return traces

In [69]:
reset_target(scope)

##inital toeplitz setup
target.write("0110100110\n")
time.sleep(0.01)
target.write("1010\n")

x2 = "1111\n"

#x0 = "10001000000010000000\n"
#x0 = "11001101000010010100\n"
#x0 = "10110100100101100101\n" #x0 = "1011 0100 1001 0110 0101\n" ##x0=[10100] x1=[01011] x2=[10010] x3=[10101]
x0 = "0110101110\n"
for i in range(4):
    warmup = get_trace(x2)

time.sleep(0.1)

target_trace = get_trace(x0)


traces0 = make_guesses(0)
#traces2 = make_guesses(2)

(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended f

Testing Hyptohesis 3
Testing Hyptohesis 1


(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.


Testing Hyptohesis 2
Testing Hyptohesis 3


(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.


...

(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.


Testing Hyptohesis 1020
Testing Hyptohesis 1021


(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.
(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.


Testing Hyptohesis 1022
Testing Hyptohesis 1023


(ChipWhisperer Scope WARNING|File OpenADC.py:932) Not intended for Husky -- just use a regular capture.


In [11]:
x_np = np.asarray(x0[:-1])
FFT(x0)

AttributeError: 'str' object has no attribute 'shape'

In [ ]:
cw.plot(warmup - target_trace)

In [70]:
MSE = []
PC = []
for i in range(1024):
    MSE.append(mean_squared_error(target_trace[0], traces0[i][0]))
    PC.append(stats.pearsonr(target_trace[0], traces0[i][0]))

    print("Guess {0}. MSE: {1}, PC: {2}".format(i,MSE[i], PC[i]))

Guess 0. MSE: 0.04330928412152202, PC: PearsonRResult(statistic=0.17692067518625038, pvalue=0.0)
Guess 1. MSE: 0.04323243467682289, PC: PearsonRResult(statistic=0.17786957792278468, pvalue=0.0)
Guess 2. MSE: 0.04341644511823517, PC: PearsonRResult(statistic=0.1744990383970891, pvalue=0.0)
Guess 3. MSE: 0.04362236518671701, PC: PearsonRResult(statistic=0.1714415017458038, pvalue=0.0)
Guess 4. MSE: 0.04359729947585483, PC: PearsonRResult(statistic=0.17156825084261437, pvalue=0.0)
Guess 5. MSE: 0.0434756158648718, PC: PearsonRResult(statistic=0.17370125583543175, pvalue=0.0)
Guess 6. MSE: 0.04375125757533229, PC: PearsonRResult(statistic=0.16808113775595415, pvalue=0.0)
Guess 7. MSE: 0.0437107783843103, PC: PearsonRResult(statistic=0.1691941130799417, pvalue=0.0)
Guess 8. MSE: 0.04359972395121233, PC: PearsonRResult(statistic=0.17058602310956802, pvalue=0.0)
Guess 9. MSE: 0.04327634254726565, PC: PearsonRResult(statistic=0.17772946357770653, pvalue=0.0)
Guess 10. MSE: 0.04311377513177273,

In [72]:
cw.plot(target_trace[0] - traces0[0][0]) * cw.plot(target_trace[0] - traces0[22][0])

:Overlay
   .Curve.I  :Curve   [x]   (y)
   .Curve.II :Curve   [x]   (y)

In [71]:
print(np.argsort(np.asarray(MSE)))

[949 439 831 ... 781 297 778]


In [ ]:
print(gen_hyp_string(27,0))